<a href="https://colab.research.google.com/github/MargaritaLantsova/FancyFASTQ_tools/blob/main/popov_seminar_ib26_tsk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Comparative Genomics of *Streptomyces albidoflavus*

**What we are going to do today:**
1. Learn what pangenomics is and why it matters (brief overview)
2. Explore eggNOG-mapper — a tool for functional genome annotation
3. Visualize and compare metabolic pathways across bacterial strains using **KEGGaNOG**

---

### Research context

*Streptomyces albidoflavus* is a soil-dwelling bacterium known for producing bioactive compounds. One strain — **SM254** — was isolated from soil and showed promising activity against *Pseudogymnoascus destructans*, the causative agent of White-nose Syndrome in bats.

To understand what makes SM254 unique at the genomic level, we compared it against 33 closely related strains using comparative genomics. This work was published in:

> Popov, I.V., Chikindas, M.L. & Popov, I.V. Genomic insights into *Streptomyces albidoflavus* SM254: tracing the putative signs of anti-*Pseudogymnoascus destructans* properties. *Braz J Microbiol* **56**, 2121–2131 (2025). https://doi.org/10.1007/s42770-025-01740-8

**Today we focus on the 6 strains from the SM254 phylogenetic clade:**

| Sample ID | Strain name |
|-----------|-------------|
| SM254 | *S. albidoflavus* SM254 (our target) |
| NBC_01747 | *S. albidoflavus* NBC 01747 |
| NBC_01673 | *S. albidoflavus* NBC 01673 |
| NBC_01665 | *S. albidoflavus* NBC 01665 |
| NBC_01621 | *S. albidoflavus* NBC 01621 |
| NBC_01110 | *S. albidoflavus* NBC 01110 |

Presentation from the lecture is available here: https://docs.google.com/presentation/d/1AG0ZlzpxD2FtWmb2gsukoX_S7R9RzhBaD9vdau1l7T0/edit?usp=sharing

---
## Part 1 (optional): Pangenomics — a bird's-eye view

A **pangenome** is the full set of genes found across a group of related bacterial strains. It consists of:

| Category | Definition | Biological meaning |
|----------|------------|--------------------|
| **Core** | Present in ALL strains | Essential housekeeping functions |
| **Soft-core** | Present in ≥95% of strains | Nearly universal, likely important |
| **Shell** | Present in some strains | Niche adaptation, mobile elements |
| **Cloud** | Present in ≤1 strain | Unique / strain-specific genes |

In our study we used **PanACoTA** to build the pangenome of 34 *S. albidoflavus* strains at 90% protein identity.

**Results:**
- Total gene families: **12,027**
- Core genome: **3,867** families

> The pangenome analysis itself takes hours to run (sequence alignment of 3867 core genes across 34 genomes). We've already done it — let's look at the results!

👉 *The instructor will show the pangenome composition charts and phylogenomic tree here.*

---
## Part 2: eggNOG-mapper — functional annotation

**eggNOG-mapper** assigns functional annotations to genes using orthology: it maps your sequences to the eggNOG database of orthologous groups and transfers annotations (KEGG pathways, GO terms, COG categories, etc.).

### Try it yourself — no installation needed!

1. Go to **[http://eggnog-mapper.embl.de](http://eggnog-mapper.embl.de)** (may be down) or **https://usegalaxy.eu/root?tool_id=eggnog_mapper**
2. Paste a protein sequence (or upload a small FASTA)
3. Choose organism group: *Bacteria*
4. Submit and explore the output table

The output file (`.emapper.annotations`) is a tab-separated table with columns like:
- `KEGG_ko` — KEGG orthology IDs
- `KEGG_Pathway` — mapped metabolic pathways
- `COG_category` — functional category letter
- `Description` — gene function description

### What we did for this study

We ran eggNOG-mapper on **whole genome sequences** of 6 strains using the command-line tool:

```bash
emapper.py -i SM254.fa --itype genome --genepred prodigal \
    -o SM254 --output_dir eggNOG/SM254/ --cpu 0
```

This predicts genes with **Prodigal** and annotates them all in one step. The results are already prepared for you below.

---
## Part 3: KEGGaNOG — metabolic pathway visualization

**KEGGaNOG** is a Python tool that takes eggNOG-mapper output and produces ready-to-publish visualizations of KEGG metabolic pathway completeness.

It wraps **KEGG-Decoder** logic and adds a clean API for heatmaps, barplots, radar plots, and correlation networks.

> GitHub: [https://github.com/iliapopov17/KEGGaNOG](https://github.com/iliapopov17/KEGGaNOG)

### Step 1 — Install KEGGaNOG

In [ ]:
import os

In [ ]:
# kegganog==1.0 was pinned here because Google Colab ships with older versions of pandas
# and other dependencies that are incompatible with newer releases.
# The latest version of KEGGaNOG is 1.5.001.1 — if you're running this locally,
# you can safely install it without the version pin:
#   pip install kegganog gdown

!pip install -q kegganog==1.0 gdown

#Экстренный комментарий:
#Google Colab будет жаловаться на конфликт версий зависимостей даже так
#Просто запустите ячейку второй раз, когда увидите ошибку
#Если не поможет – пишите, буду чинить

In [ ]:
!KEGGaNOG -V

### Step 2 — Download eggNOG-mapper annotation files
Run the cell below — all files will be downloaded automatically from Google Drive.

In [ ]:
!gdown --folder 1miiPCS4-TCFRpDWOsFCD5JoskVVRkbzt --remaining-ok -q && \
 mv popov.seminar.ib26.data/eggNOG . && \
 rm -rf popov.seminar.ib26.data

In [ ]:
samples = ["SM254", "NBC_01747", "NBC_01673", "NBC_01665", "NBC_01621", "NBC_01110"]

all_ok = True
for sample in samples:
    path = f"eggNOG/{sample}/{sample}.emapper.annotations"
    if os.path.exists(path):
        print(f"✓ {path}")
    else:
        print(f"✗ MISSING: {path}")
        all_ok = False

if all_ok:
    print("\nAll files found — ready to go!")
else:
    print("\nSome files are missing. Please upload the eggNOG/ folder.")

### Step 3 — Create a list of annotation files for KEGGaNOG

In [ ]:
!ls eggNOG/*/*.emapper.annotations > listFile.txt
!cat listFile.txt

### Step 4 — Run KEGGaNOG in multi-sample mode

This will compute KEGG pathway completeness scores for all 6 strains and produce a merged table.

In [ ]:
!KEGGaNOG -M -i listFile.txt -o kegganog_StAl --overwrite

### Step 5 — Run KEGGaNOG in single-sample mode for SM254

Single mode gives us a detailed profile for just one strain — our target SM254.

In [ ]:
!KEGGaNOG -i eggNOG/SM254/SM254.emapper.annotations -o kegganog_SM254 --overwrite

### Step 6 — Import libraries and load data

In [ ]:
import kegganog as kgn
import pandas as pd

# Multi-sample pathway completeness table
df_multi = pd.read_csv("kegganog_StAl/merged_pathways.tsv", sep="\t")

# Single-sample table for SM254
df_single = pd.read_csv("kegganog_SM254/SAMPLE_pathways.tsv", sep="\t")

print(f"Pathways in multi-sample table: {len(df_multi)}")
print(f"Pathways in SM254 table: {df_single.shape[1] - 1}")
df_multi.head()

---
### Visualization 1 — Heatmap of all pathways across 6 strains

Each cell shows the completeness of a KEGG pathway (0 = absent, 1 = complete). Pathways are grouped by functional category.

In [ ]:
sample_order = ["SM254", "NBC_01747", "NBC_01673", "NBC_01665", "NBC_01621", "NBC_01110"] #Play with the parameters as homework.
df_heat = df_multi[["Function"] + sample_order]

kgnheat = kgn.heatmap(
    df_heat,
    color="Greens", #Play with the parameters as homework.
    group=True, #Play with the parameters as homework.
)
kgnheat.plotfig()

---
### Visualization 2 — Barplot of SM254 metabolic profile

Shows completeness of all detected pathways in SM254, sorted from highest to lowest.

In [ ]:
kgnbar = kgn.barplot(
    df_single,
    figsize=(8, 14),
    sort_order="descending", #Play with the parameters as homework.
    yticks_fontsize=8,
    cmap="Blues", #Play with the parameters as homework.
)
kgnbar.plotfig()

---
### Visualization 3 — Correlation network

Strains with similar metabolic pathway profiles are connected by edges. The threshold controls how similar they need to be to form a connection (0.96 = very similar).

> **Try it:** lower the threshold (e.g., `0.90`) and see what happens to the network!

In [ ]:
kgnnet = kgn.correlation_network(
    df_multi,
    threshold=0.96, #Play here
    node_size=1750, #and here
    label_fontsize=5.75, #yep and here
    label_weight="bold",
    save_matrix="correlation_network.tsv",
    figsize=(8, 8)
)
kgnnet.plotfig()

---
### Visualization 4 — Radar plot of selected pathways

Compares 4 specific pathways across all 6 strains. Each axis is one pathway; the further from center — the more complete it is.

> **Try it:** replace the pathway names with others from the heatmap!

In [ ]:
pathways = [
    "Mixed acid: Ethanol, Acetyl-CoA to Acetylaldehyde (reversible)",
    "Asparagine",
    "Polyhydroxybutyrate synthesis",
    "Competence-related core components"
]

sample_order = ["SM254", "NBC_01747", "NBC_01673", "NBC_01665", "NBC_01621", "NBC_01110"] #you what to do

kgnradar = kgn.radarplot(
    df_multi,
    pathways=pathways,
    sample_order=sample_order,
    label_weight="bold",
    label_background="white",
    label_edgecolor="black",
    label_pad=1.2,
    yticklabels=["0.2", "", "0.6", "", "1.0"], #yeah, you know that I know that you know
    ytick_weight="bold",
    fill_alpha=0.1,
    figsize=(8, 20),
    line_style="--",
    line_width=1.75,
    legend_bbox=(1.25, 1.25)
)
kgnradar.plotfig()

## Part 4: Explore on your own!

This is a sandbox. There are no wrong answers — only more or less adventurous ones.

---

### What to do

Start with the dataset you already have. Change something: the color scheme, the threshold,
the set of pathways on the radar plot, the figure size. Make the plots look different from
the defaults. That's the baseline.

If you work with bacterial genomes yourself — bring your own data. Run eggNOG-mapper,
feed the output to KEGGaNOG, and see what comes out. The tools are the same, the story
will be yours.

---

### Grading

| Score | What to do |
|-------|------------|
| **5** | Reproduce the code from the notebook, submit the output figures. Same data, same parameters, default output. It works — that's already something. |
| **6** | Same as above + a few sentences about your results: what you got, what surprised you, what was confusing. |
| **7** | One custom visualization: different colors, threshold, pathway selection — anything visibly different from the default. Briefly explain what you changed and why. |
| **8** | Two custom visualizations. |
| **9** | All visualizations are custom. Use the notebook as a data source, not a template. |
| **10** | Install KEGGaNOG locally ([Google Drive data](https://drive.google.com/drive/folders/1UNqdrlqBSe35aSDzvsOtwUwOvLtfprEf?usp=share_link)), launch `kegganog --web`, and use the web interface. Write a short feedback: what's intuitive, what's broken, what's missing. Include a screenshot of the web interface. |
| **RESPECT+** | Skip the built-in plots entirely. Use KEGGaNOG as a parser only and build your own visualizations from scratch: ggplot2, seaborn, matplotlib, PCA, PCoA, whatever meaningful. |

> **Note:** if you bring your own bacterial genome data, run eggNOG-mapper yourself, and process it through KEGGaNOG (multi+single or single only) — that automatically counts as **10 pts** regardless of whether the plots are custom or not. Doing the full pipeline on real data is the point.

---

### Suggested explorations

**1. Find a pathway that differs between strains**  
Look at the heatmap — which pathways are present in SM254 but absent in others (or vice versa)?

**2. Change the radar plot**  
Pick 3–4 interesting pathways from the heatmap and put them on the radar plot.
What biological story do they tell?

**3. Play with the correlation network threshold**  
Try `threshold=0.90`, `0.95`, `0.99`. How does the network topology change?

**4. Use eggNOG-mapper web** *(optional)*  
Go to http://eggnog-mapper.embl.de, paste any bacterial protein sequence,
and explore the annotation output.

---

### Submission

Fill out the form → [Google Form link](https://forms.gle/2s6gZnqZYNpstoxU8)

Upload your figures and write a few sentences: what you found, what worked,
what didn't, what was confusing, what you'd like to see improved.
No minimum length. Honest > polished.